# 04 – Tren en lineær klassifikator

**Mål:** trene en lineær modell på MNIST med standard PyTorch treningsløkke og evaluere på usette bilder.
Vi bygger videre på modellen fra forrige notebook.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn


In [ ]:
from pytorch_101.utils import download_mnist

train_data, test_data = download_mnist()
print(f"Treningsbilder: {len(train_data)}, testbilder: {len(test_data)}")


## Tap og optimizer

`nn.CrossEntropyLoss()` sammenligner modellens rå skårer `[B, 10]` med riktige
klasseindekser `[B]` av typen `int64`. Den gir ett tapstall; lavere er bedre.
Ikke bruk softmax før tapsfunksjonen. Nøyaktighet teller riktige valg, mens tapet
også tar hensyn til skårene. Tapet kan derfor falle uten at nøyaktigheten endres.

En **optimizer** oppdaterer modellens parametere. Vi bruker SGD med læringsrate
0.1. Læringsraten styrer størrelsen på oppdateringene.


## Grupper til trening og evaluering

Som i notatbok 03 oppretter vi datalasterne direkte. Treningsbildene
stokkes. Testbildene brukes bare til evaluering og beholder rekkefølgen.
Ingen global frøsetting er nødvendig; målingene kan variere litt.


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)


## Oppgave 1: Sett opp MNIST-modellen

Lag `model` med `nn.Flatten()` og `nn.Linear(784, 10)`, som i notatbok 03.
Lag `loss_fn = nn.CrossEntropyLoss()` og `optimizer` med
`torch.optim.SGD`, modellens parametere og læringsrate 0.1.

<details>
<summary>Hint</summary>

Optimizer må få `model.parameters()` fra akkurat den modellen du trener.

</details>


In [ ]:
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
print(model)


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert model(torch.zeros(2, 1, 28, 28)).shape == (2, 10)


## Eksempel: én oppdatering på MNIST

De samme fem stegene gjentas for hver batch. `backward()` beregner gradienter:
hvordan tapet endres lokalt når parametrene endres. `step()` bruker dem til å
oppdatere parametrene. PyTorch gjør gradientberegningen for oss.


In [ ]:
images, labels = next(iter(train_loader))
model.train()
optimizer.zero_grad()  # 1. Nullstill gamle gradienter
scores = model(images)  # 2. Beregn skårer
loss = loss_fn(scores, labels)  # 3. Beregn tap
loss.backward()  # 4. Beregn gradienter
optimizer.step()  # 5. Oppdater parametere
print("Tap før oppdateringen:", loss.item())


## Fra én batch til flere epoker

Én **epoke** er én gjennomgang av treningsutvalget. Hver batch gir én oppdatering.
Den siste batchen kan være mindre. Derfor summerer vi `loss.item() * len(images)`
og deler på antall treningsbilder for å finne gjennomsnittstapet.


## Oppgave 2: Skriv treningsløkken

Tren modellen videre i fem epoker. Lag `training_loss = []`.
For hver epoke:

1. Kall `model.train()` og sett `loss_sum = 0.0`.
2. Gå gjennom `train_loader` og bruk de fem stegene fra eksemplet over.
3. Legg `loss.item() * len(images)` til tapsummen for hver batch.
4. Del summen på `len(train_data)`, lagre i `training_loss` og skriv ut tapet.

Kjør oppgave 1 på nytt hvis du vil starte med en ny modell og optimizer.


In [ ]:
training_loss = []
for epoch in range(5):
    model.train()
    loss_sum = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        scores = model(images)
        loss = loss_fn(scores, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(images)
    mean_loss = loss_sum / len(train_data)
    training_loss.append(mean_loss)
    print(f"Epoke {epoch + 1}: treningstap {mean_loss:.4f}")


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert len(training_loss) == 5
assert all(np.isfinite(training_loss))
assert training_loss[-1] < training_loss[0]


### Refleksjon

Oppgi første og siste treningstap. Hva antyder utviklingen?
Hva er forskjellen på én batch og én epoke?
Hva gjør `backward()` og `step()` hver for seg?


**Mitt svar:**

Et eksempel er at tapet faller fra omtrent 1,01 til 0,41. Bruk egne målte tall.
Lavere tap viser bedre tilpasning til treningsdata etter tapsfunksjonens mål,
men beviser ikke at modellen fungerer godt på nye bilder.
En batch er én gruppe bilder og gir én oppdatering; en epoke går gjennom alle
treningsbildene. `backward()` beregner gradienter, mens `step()` oppdaterer parametrene.


## Evaluering

`model.eval()` velger evalueringsmodus. `torch.no_grad()` slår av registrering
av gradienter. Vi bruker begge og oppdaterer ingen parametere på testdata.


## Oppgave 3: Evaluer på testbilder

Sett modellen i evalueringsmodus. Under `torch.no_grad()`: gå gjennom
`test_loader`, beregn skårer, velg prediksjoner og tell riktige.
Lag `test_accuracy` som antall riktige delt på antall testbilder.
Skriv ut resultatet som prosent. Vis også `training_loss` som en kurve.

<details>
<summary>Hint</summary>

Start med `num_correct = 0`. Bruk `(predictions == labels).sum().item()`.

</details>

Rapporter også `num_correct` av `len(test_data)`; sammenlign med en grov
referanse på 10 % for tilfeldig gjetting blant ti klasser.


In [ ]:
model.eval()
num_correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        scores = model(images)
        predictions = scores.argmax(dim=1)
        num_correct += (predictions == labels).sum().item()
test_accuracy = num_correct / len(test_data)
print(f"Testnøyaktighet: {test_accuracy:.1%} ({num_correct}/{len(test_data)})")
fig, ax = plt.subplots()
ax.plot(range(1, 6), training_loss, marker="o")
ax.set(xlabel="Epoke", ylabel="Gjennomsnittlig treningstap", title="Den lineære modellen lærer")
plt.show()


**Kontroll:** Kjør denne cellen når du har løst oppgaven. En feilmelding betyr at du må undersøke løsningen.


In [ ]:
assert 0 <= test_accuracy <= 1

assert test_accuracy == num_correct / len(test_data)


### Refleksjon

Hva ble testnøyaktigheten, og hva betyr den i antall riktige av 1 000 bilder?
Hva ville vært galt med å kalle `optimizer.step()` i testløkken?
Er `eval()` og `no_grad()` det samme? Forklar forskjellen.

Skriv svaret ditt i Markdown-cellen under. Bruk egne resultater der det er relevant.


**Mitt svar:**

Et eksempel er 91,4 %, altså 914 riktige av 1 000 bilder. Det er ikke
en fasit: uten fast frø varierer modellen og resultatet. Bruk din egen
målte prosent og antall riktige. Oppdateringer på testdata ville gjort
testen del av treningen. `eval()` velger modellmodus, mens `no_grad()`
hindrer gradientregistrering. Ingen av dem oppdaterer parametere.


## Videre lesing

Lenkene er på engelsk og er valgfrie oppslag eller fordypning. Du trenger ikke lese alt for å løse oppgavene.

- **Start her:** [PyTorch – Optimizing Model Parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html). Sammenlign `train_loop` og `test_loop` med dine egne løkker. Finn linjene som oppdaterer parametere, og forklar hvorfor de bare finnes i treningen.
- **Samlet eksempel:** [PyTorch – Quickstart](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html). Viser data, modell, trening og evaluering i sammenheng. Eksemplet bruker FashionMNIST og en annen modell; bruk det til å kjenne igjen mønsteret. Lagring av modellen er valgfritt viderearbeid.
